# Generation of a simple DFN with PorePy and Transfer to OGS

This is work in progress

In [32]:
import numpy as np
import porepy as pp
import ogstools as ogs

# Setting up the domain and generating a random set of circular fractures

In [33]:
mins = np.array([0.,0.,0.])
maxs = np.array([10.,10.,10.])

In [34]:
bounding_box = {'xmin': mins[0], 'xmax': maxs[0], 'ymin': mins[1], 'ymax': maxs[1], 'zmin': mins[2], 'zmax': maxs[2]}
domain = pp.Domain(bounding_box=bounding_box)
domain

pp.Domain(bounding_box={'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0})

In [35]:
nfracs = 8
r_range = np.array([3,9])
f_i = np.array([])
for i in range(nfracs):
    center = np.random.rand(3) * (maxs - mins) + mins
    major_axis = np.random.rand() * (r_range[1] - r_range[0]) + r_range[0]
    minor_axis = major_axis.copy() #circular
    major_axis_angle = 0. #for circular
    strike_angle = np.random.rand() * np.pi - np.pi/2
    dip_angle = np.random.rand() * np.pi - np.pi/2
    f_i = np.append(f_i,pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle))

In [36]:
network = pp.create_fracture_network(fractures=f_i,domain=domain)
network

Three-dimensional fracture network with 8 plane fractures.
The domain is a cuboid with bounding box: {'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0}.

## Meshing ... 

In [37]:
mesh_args = {'cell_size_boundary': 1.0, 'cell_size_fracture': 0.5, 'cell_size_min': 0.2}
mdg = pp.create_mdg("simplex", mesh_args, network)

### Here I remove 3D mesh to visualize only 2D DFN

In [38]:
mdg2d = mdg.copy()
for sd in mdg2d.subdomains():
    if sd.dim == 3:
        mdg2d.remove_subdomain(sd)
mdg2d

Mixed-dimensional grid containing 33 grids and 62 interfaces.
Maximum dimension present: 2 
Minimum dimension present: 0 
8 grids of dimension 2 with in total 6990 cells
22 grids of dimension 1 with in total 156 cells
3 grids of dimension 0 with in total 3 cells
44 interfaces between grids of dimension 2 and 1 with in total 624 mortar cells.
18 interfaces between grids of dimension 1 and 0 with in total 18 mortar cells.

In [39]:
#pp.plot_grid(mdg2d, figsize=(12,12), plot_2d=False)

## Export to VTU and import in OGS. Setting up Material IDs

In [40]:
pp.Exporter(mdg2d, 'mixed_dimensional_grid').write_vtu()

In [41]:
DFN_2D = ogs.Mesh('mixed_dimensional_grid_constant_2.vtu')
DFN_2D

Mesh (0x7f0ece4fada0)
  N Cells:    6990
  N Points:   4121
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   0.000e+00, 1.000e+01
  Z Bounds:   0.000e+00, 1.000e+01
  N Arrays:   6

In [42]:
DFN_2D['MaterialIDs'] = DFN_2D['subdomain_id'] - DFN_2D['subdomain_id'].min()

In [43]:
fig = DFN_2D.plot('MaterialIDs',show_edges=True)

Widget(value='<iframe src="http://localhost:41839/index.html?ui=P_0x7f0f2b72eb40_3&reconnect=auto" class="pyvi…